# Ejercicio 3

3. En el campus encontrará el archivo “datos_para_clustering.mat” que contiene una matriz de datos de 500 mediciones de una variable de 100 dimensiones.

    a) Utilice una red de Kohonen para reducir la dimensionalidad de los datos.
   
    b) Verifique la presencia de _clusters_, e indique cuantos puede visualizar, haciendo uso de la matriz $U$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [ ]:
np.random.seed(110007)
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern"
})

In [ ]:
mat_data = loadmat('datos_para_clustering.mat')
dataset = mat_data['datos']

In [ ]:
class KohonenNet:
    def __init__(self, n_neurons=(10, 10), dimension=2):
        self.weights = np.random.uniform(low=-1.0, high=1.0, size=(n_neurons[0], n_neurons[1], dimension))
        self.n_neurons = n_neurons
        self.grid_x, self.grid_y = np.meshgrid(np.arange(n_neurons[0]), np.arange(n_neurons[1]), indexing='ij')
        self.saved_weights = []

    def train(self, X, epochs=100, lr=0.01, sigma_sq_start=1.0, save_epochs=None):
        n_samples = X.shape[0]
        
        for epoch in range(epochs):
            idx = np.random.permutation(n_samples)
            sigma_sq = sigma_sq_start * (1 - epoch / epochs)

            if save_epochs and epoch + 1 in save_epochs:
                self.saved_weights.append(np.copy(self.weights))

            for i in idx:
                x = X[i]
                dist_sq = np.sum((self.weights - x) ** 2, axis=2)
                i_star = np.unravel_index(np.argmin(dist_sq), dist_sq.shape)
                dist_sq_grid = (self.grid_x - i_star[0])**2 + (self.grid_y - i_star[1])**2
                neighbourhood = np.exp(-dist_sq_grid / (2 * sigma_sq))
                delta_weights = lr * neighbourhood[:, :, np.newaxis] * (x - self.weights)
                self.weights += delta_weights

            if (epoch + 1) % 10 == 0:
                print(f'Epoch {epoch+1:3d}/{epochs} complete.', end='\r', flush=True)

In [ ]:
model = KohonenNet(n_neurons=(30, 30), dimension=100)
model.train(dataset, epochs=500, lr=0.1, sigma_sq_start=5.0)

In [ ]:
U = np.zeros(model.n_neurons)

for i in range(1, model.n_neurons[0] - 1):
    for j in range(1, model.n_neurons[1] - 1):
        neighbours = np.array([model.weights[i - 1, j], model.weights[i + 1, j],
                               model.weights[i, j - 1], model.weights[i, j + 1]])
        U[i, j] = np.mean(np.linalg.norm(model.weights[i, j] - neighbours))

plt.figure(figsize=(5, 4))
plt.imshow(U, origin='lower', cmap='inferno')
plt.colorbar(label='Matriz $U_{i, j}$')
plt.xlabel('Índice $i$')
plt.ylabel('Índice $j$')
plt.tight_layout()
plt.savefig('../informe/img/ej3/U.svg')
plt.show()

In [ ]:
neuron_wins = np.zeros(model.n_neurons)
for x in dataset:
    dist_sq = np.sum((model.weights - x) ** 2, axis=2)
    i_star = np.unravel_index(np.argmin(dist_sq), dist_sq.shape)
    neuron_wins[i_star] += 1

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
reduced = pca.fit_transform(dataset)

ratios = pca.explained_variance_ratio_
cum_variance = 100 * np.cumsum(ratios)

print(f'Cumulative explained variance ratio of first 2 PC: {cum_variance[0]:.2f} %, {cum_variance[1]:.2f} %.')

plt.figure(figsize=(4, 4))
plt.scatter(reduced[:, 0], reduced[:, 1])
plt.xlabel('Primera componente principal')
plt.ylabel('Segunda componente principal')
plt.grid()
plt.tight_layout()
plt.savefig('../informe/img/ej3/pca.svg')
plt.show()